##Importar Librerias

In [0]:

from pyspark.sql import functions as F
from datetime import datetime
import unicodedata

Timestamp único para toda la corrida: se usa para nombrar las
carpetas de salida de Bronze/Silver/Gold (versionado manual, ya ue Parquet no lo hace solo como Delta).


In [0]:
TIMESTAMP_EJECUCION = datetime.now().strftime("%Y-%m-%d_%H%M%S")
print(f"Timestamp de esta ejecución del pipeline: {TIMESTAMP_EJECUCION}")
 
# Los datos traen fechas en formatos mezclados A PROPÓSITO (desnormalización
# de la consigna). Con el modo ANSI de Spark activado (default en runtimes
# nuevos), intentar parsear una fecha con el formato "equivocado" tira una
# excepción y frena todo el pipeline. Lo desactivamos para que, en cambio,
# devuelva NULL en esos casos -> es lo que necesitamos para poder auditar
# cuántas fechas no matchearon ningún formato conocido.
spark.conf.set("spark.sql.ansi.enabled", "false")

Parámetros de rutas (ADF puede pasarlos vía widgets al llamar el notebook; 
los valores de acá son los defaults para poder correr el notebook suelto desde Databricks sin depender de ADF)

In [0]:
dbutils.widgets.text("ruta_landing", "abfss://integracion-agritech-central-desa@intdesa.dfs.core.windows.net/0. raw/")
dbutils.widgets.text("ruta_bronze", "abfss://integracion-agritech-central-desa@intdesa.dfs.core.windows.net/1. bronze/")
dbutils.widgets.text("ruta_silver", "abfss://integracion-agritech-central-desa@intdesa.dfs.core.windows.net/2. silver/")
dbutils.widgets.text("ruta_cuarentena", "abfss://integracion-agritech-central-desa@intdesa.dfs.core.windows.net/cuarentena/")
dbutils.widgets.text("ruta_gold", "abfss://integracion-agritech-central-desa@intdesa.dfs.core.windows.net/3. gold/")
 
RUTA_LANDING    = dbutils.widgets.get("ruta_landing")
RUTA_BRONZE     = dbutils.widgets.get("ruta_bronze")
RUTA_SILVER     = dbutils.widgets.get("ruta_silver")
RUTA_CUARENTENA = dbutils.widgets.get("ruta_cuarentena")
RUTA_GOLD       = dbutils.widgets.get("ruta_gold")
 
print(f"Landing:    {RUTA_LANDING}")
print(f"Bronze:     {RUTA_BRONZE}")
print(f"Silver:     {RUTA_SILVER}")
print(f"Cuarentena: {RUTA_CUARENTENA}")
print(f"Gold:    {RUTA_GOLD}")

## Paso 1 — Funciones utilitarias (auditoría y normalización)

In [0]:
def auditar_completo(df, titulo):
    """Auditoría completa: se usa al INICIO de cada tabla (recién cargada, o
    recién entrando a una nueva capa), antes de aplicar ninguna transformación."""
    print("=" * 80)
    print(f"AUDITORÍA COMPLETA — {titulo}")
    print(f"Registros: {df.count()}  |  Columnas: {len(df.columns)}")
    print(f"Nombres de columnas: {df.columns}")
    print("Muestra de 5 filas:")
    df.show(5, truncate=False)
    print("=" * 80)
 
 
def auditar_paso(df, paso):
    """Auditoría liviana: se usa ANTES de cada transformación individual dentro
    del procesamiento de una tabla, para trazar cómo cambia el volumen de datos
    paso a paso."""
    print(f"  [AUDITORÍA] Antes de '{paso}' -> {df.count()} registros, {len(df.columns)} columnas")
 
 
def normalizar_nombre_columna(nombre):
    """Minúsculas, sin acentos, sin espacios ni caracteres especiales."""
    nombre = nombre.strip().lower()
    nombre = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode("ascii")
    nombre = nombre.replace(" ", "_").replace("%", "pct").replace("-", "_")
    return nombre
 
 
def normalizar_columnas(df):
    """Aplica normalizar_nombre_columna a todas las columnas del DataFrame."""
    for col_original in df.columns:
        col_nueva = normalizar_nombre_columna(col_original)
        if col_nueva != col_original:
            df = df.withColumnRenamed(col_original, col_nueva)
    return df
 
 
FORMATOS_FECHA = ["yyyy-MM-dd", "dd/MM/yyyy", "dd-MM-yyyy", "yyyy/MM/dd"]
FORMATOS_TIMESTAMP = ["yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm", "yyyy-MM-dd'T'HH:mm:ss"]
 
 
def normalizar_id(columna, prefijo):
    """Unifica IDs que vienen en formatos inconsistentes desde el origen
    (desnormalización de la consigna). Ejemplos reales de un mismo tipo:
    'SEN_001', 'SEN-2', 'sen3', 'SEN0088'. Sin esto, los joins entre tablas
    fallan silenciosamente y se pierden datos en Gold.
 
    Estrategia: pasar a mayúsculas, extraer SOLO los dígitos, y reconstruir
    el ID en formato canónico PREFIJO + número con ceros a la izquierda
    (ej. -> 'SEN0001', 'SEN0002', 'SEN0003', 'SEN0088'). Si no hay dígitos,
    devuelve NULL (ID inválido)."""
    digitos = F.regexp_extract(F.col(columna), r"(\d+)", 1)
    return F.when(
        digitos != "",
        F.concat(F.lit(prefijo), F.lpad(digitos, 4, "0"))
    ).otherwise(F.lit(None))
 
 
def parsear_fecha_multiformato(columna, formatos=FORMATOS_FECHA):
    """Los CSV/JSON traen fechas en varios formatos distintos (a propósito,
    por la desnormalización pedida en la consigna). Se prueba cada formato
    y se toma el primero que matchee -> devuelve un DATE real, no texto."""
    intentos = [F.to_date(F.col(columna), fmt) for fmt in formatos]
    return F.coalesce(*intentos)
 
 
def parsear_timestamp_multiformato(columna, formatos=FORMATOS_TIMESTAMP):
    intentos = [F.to_timestamp(F.col(columna), fmt) for fmt in formatos]
    return F.coalesce(*intentos)
 
 
def ruta_con_timestamp(ruta_base, nombre_tabla):
    """Arma la ruta de salida versionada por fecha_hora de ejecución."""
    return f"{ruta_base}{nombre_tabla}/{TIMESTAMP_EJECUCION}/"
 
 
print("Funciones utilitarias definidas: auditar_completo, auditar_paso, "
      "normalizar_columnas, parsear_fecha_multiformato, "
      "parsear_timestamp_multiformato, ruta_con_timestamp")

## CAPA BRONZE
Ingesta cruda: los datos se guardan **tal cual vienen del archivo origen**, sin ninguna limpieza ni transformación. Solo cambia el formato (CSV/JSON -> Parquet).

### Bronze: agricultores

In [0]:
print("Leyendo agricultores.csv desde landing...")
bronze_agricultores = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(f"{RUTA_LANDING}agricultores.csv")
)
auditar_completo(bronze_agricultores, "Bronze - agricultores (recién leído, sin tocar)")
 
ruta_salida = ruta_con_timestamp(RUTA_BRONZE, "agricultores")
bronze_agricultores.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Bronze: parcelas

In [0]:
print("Leyendo parcelas.csv desde landing...")
bronze_parcelas = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(f"{RUTA_LANDING}parcelas.csv")
)
auditar_completo(bronze_parcelas, "Bronze - parcelas (recién leído, sin tocar)")
 
ruta_salida = ruta_con_timestamp(RUTA_BRONZE, "parcelas")
bronze_parcelas.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Bronze: sensores

In [0]:
print("Leyendo sensores.csv desde landing...")
bronze_sensores = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(f"{RUTA_LANDING}sensores.csv")
)
auditar_completo(bronze_sensores, "Bronze - sensores (recién leído, sin tocar)")
 
ruta_salida = ruta_con_timestamp(RUTA_BRONZE, "sensores")
bronze_sensores.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Bronze: mantenimientos

In [0]:
print("Leyendo mantenimientos.csv desde landing...")
bronze_mantenimientos = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(f"{RUTA_LANDING}mantenimientos.csv")
)
auditar_completo(bronze_mantenimientos, "Bronze - mantenimientos (recién leído, sin tocar)")
 
ruta_salida = ruta_con_timestamp(RUTA_BRONZE, "mantenimientos")
bronze_mantenimientos.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Bronze: lecturas (fuente REST API / GitHub, JSON)

In [0]:
print("Leyendo lecturas.json desde landing...")
bronze_lecturas = spark.read.option("multiline", "true").json(f"{RUTA_LANDING}lecturas.json")
auditar_completo(bronze_lecturas, "Bronze - lecturas (recién leído, sin tocar)")
 
ruta_salida = ruta_con_timestamp(RUTA_BRONZE, "lecturas")
bronze_lecturas.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Resumen de la carga a Bronze

In [0]:
print("=" * 80)
print("RESUMEN BRONZE — registros cargados sin ninguna transformación")
for nombre, df in [("agricultores", bronze_agricultores), ("parcelas", bronze_parcelas),
                    ("sensores", bronze_sensores), ("mantenimientos", bronze_mantenimientos),
                    ("lecturas", bronze_lecturas)]:
    print(f"  {nombre}: {df.count()} registros")
print("=" * 80)

## CAPA SILVER
### Acá se aplica toda la normalización: nombres de columna en minúsculas, eliminación de columnas redundantes, parseo de fechas a un tipo real, filtrado de valores fuera de rango, y eliminación de duplicados exactos.

### Silver: agricultores

In [0]:

auditar_completo(bronze_agricultores, "Silver - agricultores (partiendo de Bronze)")
 
auditar_paso(bronze_agricultores, "normalizar nombres de columnas")
silver_agricultores = normalizar_columnas(bronze_agricultores)
 
auditar_paso(silver_agricultores, "normalizar id_agricultor a formato canónico (AGRxxxx)")
silver_agricultores = silver_agricultores.withColumn(
    "id_agricultor", normalizar_id("id_agricultor", "AGR")
)
 
auditar_paso(silver_agricultores, "normalizar nombre y apellido (trim + capitalizar)")
silver_agricultores = silver_agricultores.withColumn(
    "nombre", F.initcap(F.trim(F.col("nombre")))
).withColumn(
    "apellido", F.initcap(F.trim(F.col("apellido")))
)
 
auditar_paso(silver_agricultores, "normalizar email (trim + minúsculas)")
silver_agricultores = silver_agricultores.withColumn(
    "email", F.lower(F.trim(F.col("email")))
)
 
auditar_paso(silver_agricultores, "parsear fecha_alta a tipo DATE")
silver_agricultores = silver_agricultores.withColumn(
    "fecha_alta_normalizada", parsear_fecha_multiformato("fecha_alta")
)
 
auditar_paso(silver_agricultores, "normalizar provincia (trim + capitalizar)")
silver_agricultores = silver_agricultores.withColumn(
    "provincia", F.initcap(F.trim(F.col("provincia")))
)
 
auditar_paso(silver_agricultores, "descartar filas sin id_agricultor válido")
silver_agricultores = silver_agricultores.filter(F.col("id_agricultor").isNotNull())
 
auditar_paso(silver_agricultores, "eliminar duplicados exactos")
silver_agricultores = silver_agricultores.dropDuplicates()
 
auditar_paso(silver_agricultores, "eliminar duplicados por id_agricultor (quedarse con 1)")
silver_agricultores = silver_agricultores.dropDuplicates(["id_agricultor"])
 
silver_agricultores = silver_agricultores.withColumn("ts_procesamiento_silver", F.current_timestamp())
 
print(f"Resultado final Silver - agricultores: {silver_agricultores.count()} registros")
ruta_salida = ruta_con_timestamp(RUTA_SILVER, "agricultores")
silver_agricultores.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Silver: parcelas
Se eliminan `nombre_agricultor` y `provincia`: son datos del agricultor duplicados a propósito en el CSV origen (desnormalización). En Silver ya no hacen falta como columnas propias — la relación queda por `id_agricultor`.

In [0]:
auditar_completo(bronze_parcelas, "Silver - parcelas (partiendo de Bronze)")
 
auditar_paso(bronze_parcelas, "normalizar nombres de columnas")
silver_parcelas = normalizar_columnas(bronze_parcelas)
 
auditar_paso(silver_parcelas, "normalizar id_parcela e id_agricultor a formato canónico")
silver_parcelas = silver_parcelas.withColumn(
    "id_parcela", normalizar_id("id_parcela", "PAR")
).withColumn(
    "id_agricultor", normalizar_id("id_agricultor", "AGR")
)
 
auditar_paso(silver_parcelas, "eliminar columnas redundantes (nombre_agricultor, provincia)")
silver_parcelas = silver_parcelas.drop("nombre_agricultor", "provincia")
 
auditar_paso(silver_parcelas, "normalizar nombre_parcela y cultivo_principal (trim + capitalizar)")
silver_parcelas = silver_parcelas.withColumn(
    "nombre_parcela", F.initcap(F.trim(F.col("nombre_parcela")))
).withColumn(
    "cultivo_principal", F.initcap(F.trim(F.col("cultivo_principal")))
)
 
auditar_paso(silver_parcelas, "parsear fecha_siembra a tipo DATE")
silver_parcelas = silver_parcelas.withColumn(
    "fecha_siembra_normalizada", parsear_fecha_multiformato("fecha_siembra")
)
 
auditar_paso(silver_parcelas, "poner en null superficie_ha fuera de rango físico (<=0)")
silver_parcelas = silver_parcelas.withColumn(
    "superficie_ha",
    F.when(F.col("superficie_ha") > 0, F.col("superficie_ha")).otherwise(F.lit(None))
)
 
auditar_paso(silver_parcelas, "eliminar duplicados exactos")
silver_parcelas = silver_parcelas.dropDuplicates()
 
auditar_paso(silver_parcelas, "descartar filas sin id_parcela válido")
silver_parcelas = silver_parcelas.filter(F.col("id_parcela").isNotNull())
 
auditar_paso(silver_parcelas, "eliminar duplicados por id_parcela (quedarse con 1)")
silver_parcelas = silver_parcelas.dropDuplicates(["id_parcela"])
 
silver_parcelas = silver_parcelas.withColumn("ts_procesamiento_silver", F.current_timestamp())
 
print(f"Resultado final Silver - parcelas: {silver_parcelas.count()} registros")
ruta_salida = ruta_con_timestamp(RUTA_SILVER, "parcelas")
silver_parcelas.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Silver: sensores
Se eliminan `nombre_parcela` y `cultivo_asociado`: duplicados de la tabla `parcelas`. La relación queda por `id_parcela`.

In [0]:

auditar_completo(bronze_sensores, "Silver - sensores (partiendo de Bronze)")
 
auditar_paso(bronze_sensores, "normalizar nombres de columnas")
silver_sensores = normalizar_columnas(bronze_sensores)
 
auditar_paso(silver_sensores, "normalizar id_sensor e id_parcela a formato canónico")
silver_sensores = silver_sensores.withColumn(
    "id_sensor", normalizar_id("id_sensor", "SEN")
).withColumn(
    "id_parcela", normalizar_id("id_parcela", "PAR")
)
 
auditar_paso(silver_sensores, "eliminar columnas redundantes (nombre_parcela, cultivo_asociado)")
silver_sensores = silver_sensores.drop("nombre_parcela", "cultivo_asociado")
 
auditar_paso(silver_sensores, "normalizar tipo_sensor y marca (trim)")
silver_sensores = silver_sensores.withColumn(
    "tipo_sensor", F.trim(F.col("tipo_sensor"))
).withColumn(
    "marca", F.trim(F.col("marca"))
)
 
auditar_paso(silver_sensores, "normalizar valores de 'estado' (trim + capitalizar)")
silver_sensores = silver_sensores.withColumn(
    "estado", F.initcap(F.trim(F.col("estado")))
)
 
auditar_paso(silver_sensores, "parsear fecha_instalacion a tipo DATE")
silver_sensores = silver_sensores.withColumn(
    "fecha_instalacion_normalizada", parsear_fecha_multiformato("fecha_instalacion")
)
 
auditar_paso(silver_sensores, "poner en null latitud/longitud fuera del rango de Argentina")
silver_sensores = silver_sensores.withColumn(
    "latitud", F.when(F.col("latitud").between(-55, -21), F.col("latitud")).otherwise(F.lit(None))
).withColumn(
    "longitud", F.when(F.col("longitud").between(-74, -53), F.col("longitud")).otherwise(F.lit(None))
)
 
auditar_paso(silver_sensores, "eliminar duplicados exactos")
silver_sensores = silver_sensores.dropDuplicates()
 
auditar_paso(silver_sensores, "descartar filas sin id_sensor válido")
silver_sensores = silver_sensores.filter(F.col("id_sensor").isNotNull())
 
auditar_paso(silver_sensores, "eliminar duplicados por id_sensor (quedarse con 1)")
silver_sensores = silver_sensores.dropDuplicates(["id_sensor"])
 
silver_sensores = silver_sensores.withColumn("ts_procesamiento_silver", F.current_timestamp())
 
print(f"Resultado final Silver - sensores: {silver_sensores.count()} registros")
ruta_salida = ruta_con_timestamp(RUTA_SILVER, "sensores")
silver_sensores.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Silver: mantenimientos
Se eliminan `tipo_sensor` y `nombre_parcela`: duplicados de la tabla `sensores`. La relación queda por `id_sensor`.

In [0]:
 
auditar_completo(bronze_mantenimientos, "Silver - mantenimientos (partiendo de Bronze)")
 
auditar_paso(bronze_mantenimientos, "normalizar nombres de columnas")
silver_mantenimientos = normalizar_columnas(bronze_mantenimientos)
 
auditar_paso(silver_mantenimientos, "normalizar id_mantenimiento e id_sensor a formato canónico")
silver_mantenimientos = silver_mantenimientos.withColumn(
    "id_mantenimiento", normalizar_id("id_mantenimiento", "MNT")
).withColumn(
    "id_sensor", normalizar_id("id_sensor", "SEN")
)
 
auditar_paso(silver_mantenimientos, "eliminar columnas redundantes (tipo_sensor, nombre_parcela)")
silver_mantenimientos = silver_mantenimientos.drop("tipo_sensor", "nombre_parcela")
 
auditar_paso(silver_mantenimientos, "normalizar tipo_mantenimiento y tecnico_responsable (trim + capitalizar)")
silver_mantenimientos = silver_mantenimientos.withColumn(
    "tipo_mantenimiento", F.initcap(F.trim(F.col("tipo_mantenimiento")))
).withColumn(
    "tecnico_responsable", F.initcap(F.trim(F.col("tecnico_responsable")))
)
 
auditar_paso(silver_mantenimientos, "normalizar observaciones (trim, y null si queda vacío)")
silver_mantenimientos = silver_mantenimientos.withColumn(
    "observaciones",
    F.when(F.trim(F.col("observaciones")) == "", F.lit(None)).otherwise(F.trim(F.col("observaciones")))
)
 
auditar_paso(silver_mantenimientos, "parsear fecha_mantenimiento a tipo DATE")
silver_mantenimientos = silver_mantenimientos.withColumn(
    "fecha_mantenimiento_normalizada", parsear_fecha_multiformato("fecha_mantenimiento")
)
 
auditar_paso(silver_mantenimientos, "poner en null costo_usd fuera de rango físico (< 0)")
silver_mantenimientos = silver_mantenimientos.withColumn(
    "costo_usd",
    F.when(F.col("costo_usd") >= 0, F.col("costo_usd")).otherwise(F.lit(None))
)
 
auditar_paso(silver_mantenimientos, "eliminar duplicados exactos")
silver_mantenimientos = silver_mantenimientos.dropDuplicates()
 
silver_mantenimientos = silver_mantenimientos.withColumn("ts_procesamiento_silver", F.current_timestamp())
 
print(f"Resultado final Silver - mantenimientos: {silver_mantenimientos.count()} registros")
ruta_salida = ruta_con_timestamp(RUTA_SILVER, "mantenimientos")
silver_mantenimientos.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Silver: lecturas
Es la tabla con más pasos: normalización + join con `sensores` (para integridad referencial) + reglas de calidad por tipo de sensor + separación de lecturas válidas / rechazadas.

In [0]:
auditar_completo(bronze_lecturas, "Silver - lecturas (partiendo de Bronze)")
 
auditar_paso(bronze_lecturas, "normalizar nombres de columnas")
silver_lecturas = normalizar_columnas(bronze_lecturas)
 
auditar_paso(silver_lecturas, "normalizar id_sensor a formato canónico (clave para el join)")
silver_lecturas = silver_lecturas.withColumn(
    "id_sensor", normalizar_id("id_sensor", "SEN")
)
 
auditar_paso(silver_lecturas, "normalizar tipo_sensor y unidad_medida (trim)")
silver_lecturas = silver_lecturas.withColumn(
    "tipo_sensor", F.trim(F.col("tipo_sensor"))
).withColumn(
    "unidad_medida", F.trim(F.col("unidad_medida"))
)
 
auditar_paso(silver_lecturas, "eliminar columna redundante (nombre_parcela)")
# nombre_parcela viene duplicado desde el sensor; se recupera via join con
# sensores -> parcelas si hiciera falta, no como columna suelta en lecturas.
silver_lecturas = silver_lecturas.drop("nombre_parcela")
 
auditar_paso(silver_lecturas, "parsear timestamp a tipo TIMESTAMP")
silver_lecturas = silver_lecturas.withColumn(
    "timestamp_normalizado", parsear_timestamp_multiformato("timestamp")
)
 
auditar_paso(silver_lecturas, "eliminar duplicados exactos")
silver_lecturas = silver_lecturas.dropDuplicates()
 
# --- Join de integridad referencial con sensores (ya limpio) ---
auditar_paso(silver_lecturas, "join con sensores (trae id_parcela + valida existencia del sensor)")
lecturas_enriquecidas = silver_lecturas.join(
    silver_sensores.select("id_sensor", "id_parcela"), on="id_sensor", how="left"
)
 
# --- Reglas de calidad ---
RANGOS_VALIDOS = {
    "Temperatura":         (-5.0, 45.0),
    "Humedad de suelo":    (5.0, 95.0),
    "Humedad ambiente":    (10.0, 100.0),
    "pH de suelo":         (4.0, 9.0),
    "Luminosidad":         (0.0, 100000.0),
    "Pluviómetro":         (0.0, 60.0),
    "Velocidad de viento": (0.0, 90.0),
}
rangos_df = spark.createDataFrame(
    [(tipo, float(minimo), float(maximo)) for tipo, (minimo, maximo) in RANGOS_VALIDOS.items()],
    ["tipo_sensor_rango", "valor_min", "valor_max"],
)
 
auditar_paso(lecturas_enriquecidas, "aplicar reglas de calidad (obligatorios + rango por tipo + integridad)")
lecturas_con_rango = lecturas_enriquecidas.join(
    F.broadcast(rangos_df), lecturas_enriquecidas.tipo_sensor == rangos_df.tipo_sensor_rango, how="left"
)
 
cond_obligatorios = F.col("id_sensor").isNotNull() & F.col("id_lectura").isNotNull()
cond_rango = F.col("valor_medido").isNull() | F.col("valor_medido").between(F.col("valor_min"), F.col("valor_max"))
cond_integridad = F.col("id_parcela").isNotNull()
 
lecturas_validadas = lecturas_con_rango.withColumn("valido", cond_obligatorios & cond_rango & cond_integridad)

## --- Motivo de rechazo: 
para que cada registro en cuarentena diga POR QUÉ fue apartado (útil para el informe: "se detectaron X registros inválidos que se trataron de tal forma"). Un mismo registro puede incumplir más de una regla a la vez, por eso se concatenan todos los motivos que apliquen.

In [0]:
lecturas_validadas = lecturas_validadas.withColumn(
    "motivo_rechazo",
    F.array_join(
        F.array_compact(F.array(
            F.when(~cond_obligatorios, F.lit("campo_obligatorio_nulo")),
            F.when(~cond_rango, F.lit("valor_fuera_de_rango")),
            F.when(~cond_integridad, F.lit("sensor_inexistente_en_dim_sensores")),
        )),
        " | ",
    ),
)
 
columnas_auxiliares = ["valido", "tipo_sensor_rango", "valor_min", "valor_max"]
silver_lecturas_validas = (
    lecturas_validadas.filter(F.col("valido")).drop(*columnas_auxiliares, "motivo_rechazo")
)
cuarentena_lecturas = lecturas_validadas.filter(~F.col("valido")).drop(*columnas_auxiliares)
 
silver_lecturas_validas = silver_lecturas_validas.withColumn("ts_procesamiento_silver", F.current_timestamp())
cuarentena_lecturas = cuarentena_lecturas.withColumn("ts_cuarentena", F.current_timestamp())
 
auditar_completo(cuarentena_lecturas, "Cuarentena - lecturas que NO pasan a Gold")
print("Detalle de motivos de rechazo:")
cuarentena_lecturas.groupBy("motivo_rechazo").count().orderBy(F.desc("count")).show(truncate=False)
 
print(f"Lecturas válidas (avanzan a Gold): {silver_lecturas_validas.count()}")
print(f"Lecturas en cuarentena (NO avanzan a Gold): {cuarentena_lecturas.count()}")
 
ruta_validas = ruta_con_timestamp(RUTA_SILVER, "lecturas")
ruta_cuarentena = ruta_con_timestamp(RUTA_CUARENTENA, "lecturas")
silver_lecturas_validas.write.format("parquet").mode("append").save(ruta_validas)
cuarentena_lecturas.write.format("parquet").mode("append").save(ruta_cuarentena)
print(f"Guardado válidas (Silver) en: {ruta_validas}")
print(f"Guardado cuarentena en: {ruta_cuarentena}")

### Resumen de la capa Silver

In [0]:
print("=" * 80)
print("RESUMEN SILVER — tras normalización, limpieza y reglas de calidad")
for nombre, df in [("agricultores", silver_agricultores), ("parcelas", silver_parcelas),
                    ("sensores", silver_sensores), ("mantenimientos", silver_mantenimientos),
                    ("lecturas (válidas, avanzan a Gold)", silver_lecturas_validas),
                    ("lecturas (en cuarentena, NO avanzan)", cuarentena_lecturas)]:
    print(f"  {nombre}: {df.count()} registros")
print("=" * 80)

## CAPA GOLD
Agregaciones orientadas al análisis de negocio: promedio de cada magnitud medida por parcela, costo de mantenimiento acumulado, cantidad de sensores activos, y una columna de alerta para uso gerencial.

### Gold: resumen_parcelas

In [0]:
auditar_completo(silver_lecturas_validas, "Gold - resumen_parcelas (partiendo de Silver)")
 
auditar_paso(silver_lecturas_validas, "pivotear promedio de valor_medido por tipo_sensor")
promedios_por_tipo = (
    silver_lecturas_validas.groupBy("id_parcela")
    .pivot("tipo_sensor", list(RANGOS_VALIDOS.keys()))
    .agg(F.round(F.avg("valor_medido"), 2))
)
renombres = {
    "Temperatura": "temp_promedio_c",
    "Humedad de suelo": "humedad_suelo_promedio_pct",
    "Humedad ambiente": "humedad_ambiente_promedio_pct",
    "pH de suelo": "ph_promedio",
    "Luminosidad": "luminosidad_promedio_lux",
    "Pluviómetro": "lluvia_promedio_mm",
    "Velocidad de viento": "viento_promedio_kmh",
}
for original, nuevo in renombres.items():
    if original in promedios_por_tipo.columns:
        promedios_por_tipo = promedios_por_tipo.withColumnRenamed(original, nuevo)
 
auditar_paso(silver_lecturas_validas, "calcular cantidad de lecturas válidas por parcela")
cantidad_lecturas = silver_lecturas_validas.groupBy("id_parcela").agg(
    F.count("*").alias("cant_lecturas_validas")
)
 
auditar_paso(cuarentena_lecturas, "calcular cantidad de lecturas en cuarentena por parcela")
rechazos_por_parcela = cuarentena_lecturas.groupBy("id_parcela").agg(
    F.count("*").alias("cant_lecturas_rechazadas")
)
 
auditar_paso(silver_mantenimientos, "calcular costo total de mantenimiento por parcela (via join con sensores)")
costos_mantenimiento = (
    silver_mantenimientos.join(silver_sensores.select("id_sensor", "id_parcela"), on="id_sensor", how="left")
    .groupBy("id_parcela")
    .agg(F.round(F.sum("costo_usd"), 2).alias("costo_total_mantenimiento_usd"))
)
 
auditar_paso(silver_sensores, "calcular cantidad de sensores activos por parcela")
sensores_activos_por_parcela = (
    silver_sensores.filter(F.col("estado") == "Activo")
    .groupBy("id_parcela")
    .agg(F.count("*").alias("cant_sensores_activos"))
)
 
auditar_paso(silver_parcelas, "unir todo en la tabla Gold + calcular columna de riesgo")
gold_resumen_parcelas = (
    silver_parcelas.select("id_parcela", "nombre_parcela", "cultivo_principal")
    .join(promedios_por_tipo, on="id_parcela", how="left")
    .join(cantidad_lecturas, on="id_parcela", how="left")
    .join(rechazos_por_parcela, on="id_parcela", how="left")
    .join(costos_mantenimiento, on="id_parcela", how="left")
    .join(sensores_activos_por_parcela, on="id_parcela", how="left")
    .fillna(0, subset=["cant_lecturas_validas", "cant_lecturas_rechazadas",
                        "costo_total_mantenimiento_usd", "cant_sensores_activos"])
    .withColumn(
        "riesgo_calidad_datos",
        F.when(
            (F.col("cant_lecturas_rechazadas") /
             (F.col("cant_lecturas_validas") + F.col("cant_lecturas_rechazadas") + F.lit(0.0001))) > 0.2,
            F.lit(True),
        ).otherwise(F.lit(False)),
    )
    .withColumn("ts_generacion_gold", F.current_timestamp())
)
 
print(f"Resultado final Gold - resumen_parcelas: {gold_resumen_parcelas.count()} registros")
ruta_salida = ruta_con_timestamp(RUTA_GOLD, "resumen_parcelas")
gold_resumen_parcelas.write.format("parquet").mode("append").save(ruta_salida)
print(f"Guardado en: {ruta_salida}")

### Salida visual para nivel gerencial

In [0]:
display(gold_resumen_parcelas.orderBy(F.desc("riesgo_calidad_datos"), F.desc("costo_total_mantenimiento_usd")))

In [0]:
gold_resumen_parcelas = spark.read.parquet(
    "abfss://integracion-agritech-central-desa@intdesa.dfs.core.windows.net/3. gold/resumen_parcelas/*"
)

In [0]:

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
 
# Convertir Gold a Pandas para graficar (volumen chico, no hay problema de memoria)
gold_pd = gold_resumen_parcelas.toPandas()
 
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Dashboard gerencial — Resumen por parcela", fontsize=16, fontweight="bold", y=0.98)
fig.patch.set_facecolor("white")
 
# ─── Gráfico 1: Top 10 parcelas por costo de mantenimiento ───
ax1 = axes[0, 0]
top_costo = gold_pd.nlargest(10, "costo_total_mantenimiento_usd")
colores_costo = ["#E24B4A" if r else "#378ADD" for r in top_costo["riesgo_calidad_datos"]]
ax1.barh(top_costo["nombre_parcela"], top_costo["costo_total_mantenimiento_usd"], color=colores_costo)
ax1.set_xlabel("Costo total (USD)")
ax1.set_title("Top 10 parcelas por costo de mantenimiento", fontweight="bold")
ax1.invert_yaxis()
ax1.xaxis.set_major_formatter(ticker.FormatStrFormatter("$%.0f"))
# Leyenda manual
ax1.barh([], [], color="#E24B4A", label="Con riesgo de calidad")
ax1.barh([], [], color="#378ADD", label="Sin riesgo")
ax1.legend(loc="lower right", fontsize=9)
 
# ─── Gráfico 2: Distribución de lecturas válidas vs cuarentena ───
ax2 = axes[0, 1]
total_validas = gold_pd["cant_lecturas_validas"].sum()
total_cuarentena = gold_pd["cant_lecturas_rechazadas"].sum()
sizes = [total_validas, total_cuarentena]
labels_pie = [f"Válidas\n{total_validas} ({100*total_validas/(total_validas+total_cuarentena):.1f}%)",
              f"Cuarentena\n{total_cuarentena} ({100*total_cuarentena/(total_validas+total_cuarentena):.1f}%)"]
colors_pie = ["#1D9E75", "#E24B4A"]
wedges, texts = ax2.pie(sizes, labels=labels_pie, colors=colors_pie, startangle=90,
                         textprops={"fontsize": 11})
ax2.set_title("Calidad de datos — lecturas procesadas", fontweight="bold")
 
# ─── Gráfico 3: Promedios de temperatura por cultivo ───
ax3 = axes[1, 0]
por_cultivo = gold_pd.groupby("cultivo_principal").agg(
    temp_prom=("temp_promedio_c", "mean"),
    hum_prom=("humedad_suelo_promedio_pct", "mean"),
    parcelas_count=("id_parcela", "count")
).dropna(subset=["temp_prom"]).sort_values("temp_prom", ascending=True)
barras = ax3.barh(por_cultivo.index, por_cultivo["temp_prom"], color="#D85A30")
ax3.set_xlabel("Temperatura promedio (°C)")
ax3.set_title("Temperatura promedio por tipo de cultivo", fontweight="bold")
for barra, val in zip(barras, por_cultivo["temp_prom"]):
    ax3.text(val + 0.3, barra.get_y() + barra.get_height()/2, f"{val:.1f}°C",
             va="center", fontsize=9, color="#4A1B0C")
 
# ─── Gráfico 4: Sensores activos vs total de parcelas ───
ax4 = axes[1, 1]
con_sensores = (gold_pd["cant_sensores_activos"] > 0).sum()
sin_sensores = (gold_pd["cant_sensores_activos"] == 0).sum()
categorias = ["Con sensores\nactivos", "Sin sensores\nactivos"]
valores = [con_sensores, sin_sensores]
colores_sen = ["#1D9E75", "#B4B2A9"]
bars = ax4.bar(categorias, valores, color=colores_sen, width=0.5)
ax4.set_ylabel("Cantidad de parcelas")
ax4.set_title("Cobertura de sensores activos por parcela", fontweight="bold")
for bar, val in zip(bars, valores):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha="center", fontweight="bold", fontsize=12)
 
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Resumen final para el informe técnico

In [0]:
total_lecturas = silver_lecturas_validas.count() + cuarentena_lecturas.count()
print("=" * 80)
print("RESUMEN DE CALIDAD DE DATOS — pipeline completo")
print(f"Ejecución: {TIMESTAMP_EJECUCION}")
print(f"Total de lecturas procesadas: {total_lecturas}")
print(f"Lecturas válidas (avanzaron a Gold): {silver_lecturas_validas.count()} "
      f"({round(100 * silver_lecturas_validas.count() / total_lecturas, 1)}%)")
print(f"Lecturas en cuarentena (NO avanzaron a Gold): {cuarentena_lecturas.count()} "
      f"({round(100 * cuarentena_lecturas.count() / total_lecturas, 1)}%)")
print(f"Parcelas con riesgo de calidad de datos (>20% en cuarentena): "
      f"{gold_resumen_parcelas.filter(F.col('riesgo_calidad_datos')).count()}")
print("=" * 80)